In [265]:
from typing import Optional
from ollama import chat # type: ignore
from pydantic import BaseModel, Field # type: ignore
import json
import pandas as pd
import glob

from collections import defaultdict

# from itertools import product
import unicodedata
from unidecode import unidecode

import tiktoken
import re


# Perform chunking on manuscript.txt

In [275]:
import argparse
import json
import os
import re

def split_text_into_windows(corpus, window_size=500, overlap=250, document_name="doc"):
    results = []
    step_size = window_size - overlap

    for start_idx in range(0, len(corpus), step_size):
        end_idx = min(start_idx + window_size, len(corpus))

        extracted_text = corpus[start_idx:end_idx]

        results.append({
            "extracted_text": extracted_text,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "index_type": "character",
            "document_name": document_name
        })

        if end_idx == len(corpus):
            break

    return results

In [355]:
ds_name = "adipose_Emont2022"
paper_fn = f"/Users/sinabooeshaghi/projects/llmarkers/data/{ds_name}/manuscript/manuscript.txt"

with open(paper_fn, "r") as f:
    paper = f.read().strip()

# Split the text into windows
windows = split_text_into_windows(paper, window_size=250, overlap=100, document_name=f"{ds_name}")

In [356]:
len(windows)

442

In [357]:
idx = 54
paper[windows[idx]["start_idx"]:windows[idx]["end_idx"]] == windows[idx]["extracted_text"]

True

# Run LLM check on each chunk

## Marker Gene Extraction

In [358]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Permissive class object (allows for missing fields)
class GeneFinder(BaseModel):
    hasCelltypeGene: bool = Field(False, description="Does the text discuss biological cell type marker genes?")

In [359]:
def extract_genes(user_content, system_prompt, data_model, model="llama3.2"):
    response = chat(
        messages=[
            {
                'role': 'system',
                'content': system_prompt,
            },
            {
                'role': 'user',
                'content': user_content,
            }
        ],
        model = model,
        format = data_model.model_json_schema(),
        options={'temperature': 0},  # Make responses more deterministic

    )
    genes = data_model.model_validate_json(response.message.content)
    return genes.model_dump()

system_prompt = """
You are an expert biologist specialized in identifying discussions about cell types and their associated marker genes in scientific texts.
"""
user_content = f""""
Top marker genes for these revealed that VC08 (BLOOD VASCULAR CELLS) displayed a considerable overlap with the herein-identified M2 (MYC08) (MONOCYTES AND MACROPHAGES) (Jaccard index: 0.33), indicating that they may represent cells with similar function or intermediate cell states. Two of the shared marker genes were TIMP1 and ITLN1 which encode secreted proteins inhibiting neovascularization
"""

def mkusercontent(text):
    return f"""
Does the following text discuss relationships between cell types and their marker genes?
Criteria for “True”:
- Mentions or implies cell types AND specifically mentions a gene or set of genes.
- Discusses marker genes as identifying, characterizing, or differentiating particular cell types.

Otherwise, return “False”.

Text:
“””
{text}
“””
"""

In [360]:
check = []
for idx, window in enumerate(windows):

    genes = extract_genes(mkusercontent(window["extracted_text"]), system_prompt, GeneFinder, model="llama3.2")
    if idx % 50 == 0:
        print(f"Processed {idx} rows.")
    window.update({"hasCelltypeGene": genes["hasCelltypeGene"]})
    check.append(window)

Processed 0 rows.
Processed 50 rows.
Processed 100 rows.
Processed 150 rows.
Processed 200 rows.
Processed 250 rows.
Processed 300 rows.
Processed 350 rows.
Processed 400 rows.


# Filter chunks

In [361]:
filt = [i for i in check if i["hasCelltypeGene"]]

In [362]:
len(filt)

191

# Merge adjacent chunks

In [363]:
merged = []

current = filt[0]

for next_row in filt[1:]:
    if current['end_idx'] > next_row['start_idx']:
        # Calculate the non-overlapping part
        overlap_len = current['end_idx'] - next_row['start_idx']
        current['extracted_text'] += next_row['extracted_text'][overlap_len:]
        current['end_idx'] = next_row['end_idx']
    else:
        merged.append(current)
        current = next_row.copy()

merged.append(current)

# Verify position of chunks

In [364]:
print(len(merged) == sum([
    paper[m['start_idx']:m['end_idx']] == m['extracted_text'] for m in merged
]))

True


In [365]:
len(merged)

64

In [367]:
merged[0]

{'extracted_text': 'Main \n \nAn atlas of human white adipose tissue \n \nMature adipocytes are too large and fragile to withstand traditional single-cell approaches. As a result, several groups have focused on the non-adipocyte stromal-vascular fraction (SVF) of mouse^(3,4,5,6) and human7 adipose tissue. An alternative strategy involves single-nucleus sequencing (sNuc-seq), which can capture adipocytes, and has been used to describe mouse white^(8,9) and human brown adipose tissue10. To compare these approaches in the context of human white adipose tissue (WAT), we pursued experiments on two cohorts of subjects. In the first, we collected subcutaneous WAT from nine women, isolated single cells from the SVF using c',
 'start_idx': 0,
 'end_idx': 700,
 'index_type': 'character',
 'document_name': 'adipose_Emont2022',
 'hasCelltypeGene': True}

In [ ]:
def reformat_merged(merged):
    reformatted = []
    for m in merged:
        reformatted.append({
            "extracted": {
                "organism": "",
                "cell_type_label": "",
                "cell_source": "",
                "cell_state": "",
                "feature_name": "",
                "feature_type": "",
            },
            "derived": {
                "organism": "",
                "cell_type_id": "",
                "cell_type_label": "",
                "cell_source": "",
                "cell_state": "",
                "feature_name": "",
                "feature_type": "",
                "feature_identifier": "",
                "feature_identifier_type": "",
            },
            "source": {
                "source_rationale": m["extracted_text"],
                "source_type": "text",
                "source_id": "text"
            }
        })
    return reformatted
reformatted = reformat_merged(merged)